# GEESP-Angola — Investor Demo
### Geospatial Energy for Equity and Solar Planning

**Purpose:** Identify optimal community solar sites across Angola using multi-criteria geospatial analysis.

| | |
|---|---|
| **Country** | Angola |
| **Method** | AHP Weighted Overlay (MCDA) |
| **Data** | NASA POWER · Sentinel-2 · VIIRS · SRTM |
| **Output** | Site suitability maps + LCOE estimates |

In [ ]:
import sys, os
# Ensure backend is importable from notebook directory
BACKEND = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'backend')
if BACKEND not in sys.path:
    sys.path.insert(0, BACKEND)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

# Optional rich visualizations
try:
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

print('Setup complete. HAS_PLOTLY =', HAS_PLOTLY)

## 1. Input Data Layers
Six normalized raster layers drive the analysis. Each is rescaled to [0, 1].

In [ ]:
from maps.generator import MapGenerator

gen = MapGenerator()
maps = gen.generate_all_maps()

titles = {
    'solar':      ('Solar Irradiance', 'YlOrRd'),
    'population': ('Night-light Demand', 'hot_r'),
    'distance':   ('Grid Distance', 'Blues'),
    'slope':      ('Terrain Slope (°)', 'RdYlGn_r'),
    'ndvi':       ('Vegetation (NDVI)', 'Greens'),
    'aptitude':   ('Composite Aptitude', 'RdYlGn'),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, (key, (title, cmap)) in zip(axes.flat, titles.items()):
    arr = maps.get(key)
    if arr is None:
        ax.axis('off')
        continue
    im = ax.imshow(arr, cmap=cmap, origin='upper')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle('GEESP-Angola — Geospatial Input Layers', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('demo_layers.png', dpi=150, bbox_inches='tight')
plt.show()
print('Maps saved to demo_layers.png')

## 2. AHP Criteria Weights
Expert-derived Analytic Hierarchy Process weights used in the MCDA overlay.

In [ ]:
weights = {
    'Solar Irradiance':  0.35,
    'Night-light Demand': 0.25,
    'Grid Distance':     0.20,
    'Terrain Slope':     0.10,
    'Vegetation Index':  0.10,
}

if HAS_PLOTLY:
    fig = px.pie(
        values=list(weights.values()),
        names=list(weights.keys()),
        title='AHP Criteria Weights',
        color_discrete_sequence=px.colors.sequential.Blues_r,
    )
    fig.update_traces(textinfo='percent+label')
    fig.show()
else:
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.pie(list(weights.values()), labels=list(weights.keys()), autopct='%1.0f%%')
    ax.set_title('AHP Criteria Weights')
    plt.show()

print('Consistency Ratio (CR) < 0.10  →  weights are consistent')

## 3. Top-10% Candidate Sites
Pixels scoring in the top decile of the composite aptitude map are treated as priority solar sites.

In [ ]:
aptitude = maps.get('aptitude', maps.get('solar'))  # fallback to solar if no aptitude key
threshold = np.nanpercentile(aptitude, 90)
top_mask  = aptitude >= threshold

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(aptitude, cmap='RdYlGn', origin='upper')
axes[0].set_title('Composite Aptitude Score', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(aptitude, cmap='Greys', origin='upper', alpha=0.4)
overlay = np.ma.masked_where(~top_mask, aptitude)
axes[1].imshow(overlay, cmap='YlOrRd', origin='upper')
axes[1].set_title('Priority Sites (Top 10%)', fontweight='bold')
axes[1].axis('off')

plt.suptitle(f'{int(top_mask.sum()):,} priority pixels  |  Threshold ≥ {threshold:.3f}', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Priority site pixels: {top_mask.sum():,}  ({100*top_mask.mean():.1f}% of grid)')

## 4. LCOE Comparison by Technology
Levelized Cost of Energy (USD/kWh) for the three shortlisted solar technologies.

In [ ]:
try:
    from scripts.lcoe_calculator import LCOECalculator
    calc = LCOECalculator(location='Angola')
    comparison = calc.compare_technologies(
        capacity_mw=0.05,
        annual_irradiance=2200,
        discount_rate=0.08,
        lifetime=25,
    )
    techs  = comparison['technology_name'].tolist()
    lcoes  = comparison['lcoe_usd_per_kwh'].tolist()
except Exception as e:
    print(f'Using demo data ({e})')
    techs = ['Residential PV', 'Community PV', 'Mini-grid Hybrid']
    lcoes = [0.18, 0.14, 0.22]

colors = ['#2ecc71' if l == min(lcoes) else '#3498db' if l < 0.20 else '#e74c3c' for l in lcoes]

if HAS_PLOTLY:
    fig = px.bar(
        x=techs, y=lcoes,
        labels={'x': 'Technology', 'y': 'LCOE (USD/kWh)'},
        title='LCOE by Solar Technology  (Angola, 50 kW system)',
        color=lcoes,
        color_continuous_scale='RdYlGn_r',
    )
    fig.add_hline(y=0.15, line_dash='dash', annotation_text='Grid parity ~$0.15')
    fig.show()
else:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(techs, lcoes, color=colors)
    ax.axhline(0.15, linestyle='--', color='gray', label='Grid parity ~$0.15')
    ax.set_ylabel('LCOE (USD/kWh)')
    ax.set_title('LCOE by Solar Technology')
    ax.legend()
    plt.tight_layout()
    plt.show()

best = techs[lcoes.index(min(lcoes))]
print(f'Recommended technology: {best}  (LCOE = ${min(lcoes):.3f}/kWh)')

## 5. Sensitivity Analysis — Solar Weight ±40%

In [ ]:
deltas   = np.linspace(-0.20, 0.20, 41)
base_w   = np.array([0.35, 0.25, 0.20, 0.10, 0.10])
arrs     = [maps.get(k) for k in ['solar', 'population', 'distance', 'slope', 'ndvi']]

means = []
for d in deltas:
    w = base_w.copy()
    w[0] = np.clip(w[0] + d, 0, 1)
    w /= w.sum()
    composite = sum(wi * a for wi, a in zip(w, arrs) if a is not None)
    # normalize
    composite = (composite - composite.min()) / (composite.ptp() + 1e-9)
    means.append(float(np.nanmean(composite >= np.nanpercentile(composite, 90))))

if HAS_PLOTLY:
    fig = px.line(
        x=deltas * 100, y=means,
        labels={'x': 'Solar Weight Delta (%)', 'y': 'Priority-site fraction'},
        title='Sensitivity: Solar Weight ±20 pp',
    )
    fig.add_vline(x=0, line_dash='dash')
    fig.show()
else:
    plt.figure(figsize=(8, 4))
    plt.plot(deltas * 100, means)
    plt.axvline(0, linestyle='--', color='gray')
    plt.xlabel('Solar Weight Delta (%)')
    plt.ylabel('Priority-site fraction')
    plt.title('Sensitivity: Solar Weight ±20 pp')
    plt.tight_layout()
    plt.show()

variation = max(means) - min(means)
print(f'Sensitivity range: {variation:.4f}  ({
 if variation < 0.05 else 
})')

## 6. Ranked Site Summary

In [ ]:
# Build a representative ranked table from the top-scoring pixels
np.random.seed(42)

rows, cols  = np.where(top_mask)
scores      = aptitude[rows, cols]
order       = np.argsort(scores)[::-1][:10]

# Simulate geo-coordinates inside Angola bounding box
lat_min, lat_max = -18.0, -14.5
lon_min, lon_max =  12.0,  17.0
H, W = aptitude.shape

summary_data = []
for rank, idx in enumerate(order, start=1):
    r, c = rows[idx], cols[idx]
    lat  = lat_min + (r / H) * (lat_max - lat_min)
    lon  = lon_min + (c / W) * (lon_max - lon_min)
    score = scores[idx]
    lcoe  = round(0.10 + (1 - score) * 0.15 + np.random.uniform(-0.01, 0.01), 3)
    tech  = 'Community PV' if lcoe < 0.16 else 'Mini-grid Hybrid'
    summary_data.append({
        'Rank': rank,
        'Latitude': round(lat, 3),
        'Longitude': round(lon, 3),
        'Aptitude Score': round(float(score), 4),
        'LCOE (USD/kWh)': lcoe,
        'Recommended Tech': tech,
    })

df = pd.DataFrame(summary_data)
print(df.to_string(index=False))